# Day 7 — LightGBM：GBDTの実戦版

## なぜ乗り換える？（Day5までは sklearn の GradientBoosting）

1. **問題**：本物のKaggleデータは2000件じゃなく50万〜100万件。決定木が最良の分割を探すには「列のしきい値を片っ端から試す」必要があり（Day2）、候補数は最大で件数ぶん → 大データで激遅。
2. **LightGBMの解（武器①: histogram）**：木を建てる前に、各連続列を **255個の箱(bin)** に丸めてしまう。試すしきい値は常に **255通りに固定**。Nが増えても候補は増えない。
3. **実測**：10万行で sklearn GBDT 95秒 vs LightGBM 1.1秒（**約85倍**）、精度はほぼ同じ。丸めて捨てた解像度には予測に使える情報がほぼ無い（木はYES/NOの粗いカットしかしない＝23.41歳と23.43歳で切っても同じ人が左右に分かれるだけ）。

→ 速さは「精度を上げるための試行回数」に化ける。だからKaggleの定番。

**残り2つの武器（この後やる）**
- **武器②: leaf-wise** … 木の育て方。一番得する葉だけを深掘りする（`num_leaves` で制御）。
- **武器③: early stopping** … 「木は何本がベスト？」をDay5で手探りした問題を自動で解決する。

In [ ]:
# 足場：import とデータ読み込み（今日の本筋ではないので埋めてある）
import pandas as pd
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier

df = pd.read_csv('../../data/chat_sessions_complex.csv')
X = df.drop(columns=['deepened'])
y = df['deepened']
X.shape, y.value_counts().to_dict()

## 本筋①：初めてのLightGBM

お題：複雑データで cv=10 の精度を出し、Day5 と比べる。

| モデル | cv精度 |
|---|---|
| RF（Day4/5） | 0.795 |
| sklearn GBDT（Day5, n=300, lr=0.1） | 0.806 |
| **LightGBM（今日）** | ??? |

**予想を先に書く**：LightGBMはこの2つに対してどう出る？ → 

ヒント：
- API は sklearn GBDT とそっくり。`LGBMClassifier(...)` を作って `cross_val_score(model, X, y, cv=10)` の平均。
- 比較を公平にするなら Day5 と条件を揃える（`n_estimators`, `learning_rate`）。`verbose=-1` でログを黙らせる。

In [ ]:
# === 本筋：ここは自分で書く（穴あき）===
# 1) model = LGBMClassifier( ... )     # n_estimators / learning_rate / random_state=0 / verbose=-1
# 2) scores = cross_val_score(model, X, y, cv=10)
# 3) scores.mean() を表示して、上の表の ??? を埋める

# TODO


## 次：武器②leaf-wise と 武器③early stopping

- まず上の生LightGBMの数字を確定させてから進む。
- early stopping は「Day5で手探りした“木の本数”を自動で決める」体験につなげる。